# tp-05i NN の再調整（新しい特徴量セット上で）

**実行場所: ホスト PC（stg）。** tp-05g で追加した特徴量（追い切り評価・厩舎コメント・血統×条件）を入れた NN（`nn_extras_ped`）は、アンサンブル v2 で最も重い重み（win 0.475）を持った。
ところがそのパラメータは、追加前の特徴量で調整した値のまま。特徴量の数と性質が変わったので、**valid 2022・2023 だけで Optuna 調整（30 試行）→ 2024 は確認専用**で調整し直す。

- 探索空間は tp-05d より広め（隠れ層の形・エポック・学習率・重み減衰・ドロップアウト・バッチ）。
- 採否: `nn_extras_ped`（調整前）との比較で、p_win の LogLoss の差の 95% 上限 < 0、連対・複勝が悪化しない（3 fold × 3 seed、レース単位 paired bootstrap）。
- 採用なら OOF を `oof/model_oof/nn_extras_ped_tuned.parquet` に保存し、tp-05h を再実行してアンサンブルに加える。

In [ ]:
import os, sys
from pathlib import Path


def _find_root():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / "AGENTS.md").exists() and (q / "src").is_dir():
            return q
    raise RuntimeError("リポジトリルートが見つかりません")


REPO_ROOT = _find_root()
os.chdir(REPO_ROOT)
os.environ.setdefault("POLARS_MAX_THREADS", "6")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from src.utils.project_env import load_project_dotenv
load_project_dotenv()

In [ ]:
import json, subprocess, gc, time
import numpy as np
import pandas as pd
import mlflow
import optuna

from src.pipeline.mlflow.train_utils import require_tracking_server
from src.pipeline.models import tp05_data, win_models as wm, zoo_runner
from src.pipeline.models.compute import detect_compute

WORK = REPO_ROOT / "data/local/modeling/training_pipeline"
OOF_DIR = WORK / "oof/model_oof"
OUT = WORK / "improve_i"
(OUT / "optuna").mkdir(parents=True, exist_ok=True)
THREADS = detect_compute(reserve_cores=6).threads
mlflow.set_experiment("keiba-tp-hpo")
require_tracking_server()
GIT_COMMIT = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True, cwd=REPO_ROOT).stdout.strip()
D = tp05_data.load_dataset(WORK)
df, meta, folds = D["df"], D["meta"], D["folds"]
EP, EXTRAS, PED = tp05_data.load_extras_ped(df, WORK)
X = pd.concat([df[D["FEATS"]["base+S+C1"]], EP], axis=1)
TUNE_FOLDS = folds[:2]
print(f"行 {len(X):,} / 特徴量 {X.shape[1]} / 調整 fold {[f.fold_id for f in TUNE_FOLDS]} / threads {THREADS}")

In [ ]:
optuna.logging.set_verbosity(optuna.logging.WARNING)
def space(t):
    h = t.suggest_categorical("hidden", ["64-32", "128-64", "256-128", "128", "256-128-64"])
    return {"hidden": tuple(int(x) for x in h.split("-")), "epochs": t.suggest_int("epochs", 6, 24), "lr": t.suggest_float("lr", 5e-4, 5e-3, log=True),
            "wd": t.suggest_float("wd", 1e-5, 1e-2, log=True), "dropout": t.suggest_float("dropout", 0.0, 0.4), "batch_races": t.suggest_categorical("batch_races", [64, 128, 256])}
ref_res, _ = zoo_runner.run_oof("nn_ref", "nn", X, meta, TUNE_FOLDS, [0], THREADS, params={"hidden": (256, 128), "epochs": 14})
ref = float(ref_res["logloss_win"].mean())
study = optuna.create_study(study_name="tp05i_nn", storage=f"sqlite:///{OUT / 'optuna/nn.db'}", load_if_exists=True, direction="minimize", sampler=optuna.samplers.TPESampler(seed=0))
def objective(t):
    p = space(t)
    res, _ = zoo_runner.run_oof("nn_trial", "nn", X, meta, TUNE_FOLDS, [0], THREADS, params=p)
    gc.collect()
    return float(res["logloss_win"].mean())
t0 = time.time()
todo = 30 - len([t for t in study.trials if t.state.name == "COMPLETE"])
if todo > 0:
    study.optimize(objective, n_trials=todo)
print(f"参照(256-128, 14 epoch, 既定) 調整 fold の win LogLoss = {ref:.5f} / 最良 {study.best_value:.5f}（{time.time() - t0:.0f}s）:", study.best_params)
BEST = study.best_params
BEST_P = {**{k: v for k, v in BEST.items() if k != "hidden"}, "hidden": tuple(int(x) for x in BEST["hidden"].split("-"))}
(OUT / "best_params.json").write_text(json.dumps({"params": BEST, "ref": ref, "best": study.best_value}, ensure_ascii=False, indent=1))

In [ ]:
# 全 fold（2022〜2024）× 3 seed で OOF を保存し、調整前（nn_extras_ped）と比べる
key = ["race_id", "horse_id"]
with mlflow.start_run(run_name="tp-05i-nn_extras_ped_tuned", tags={"notebook": "tp-05i", "git_commit": GIT_COMMIT, "base": "nn_extras_ped"}):
    res, oof = zoo_runner.run_oof("nn_extras_ped_tuned", "nn", X, meta, folds, [0, 1, 2], THREADS, params=dict(BEST_P), out_dir=OOF_DIR)
    mlflow.log_params({k: str(v) for k, v in BEST_P.items()})
    mlflow.log_metrics({f"mean_{c}": float(res[c].mean()) for c in res.columns if c not in ("model", "fold")})
display(res.round(5))
fin_map = meta.set_index(key)["finish_position"]
b = pd.read_parquet(OOF_DIR / "nn_extras_ped.parquet")
m = b.merge(oof, on=key, suffixes=("_b", "_v"))
fin = fin_map.loc[list(zip(m["race_id"], m["horse_id"]))].to_numpy()
out = {"variant": "nn_extras_ped_tuned", "基準": "nn_extras_ped"}
for tgt, k in (("win", 1), ("top2", 2), ("top3", 3)):
    y = (fin <= k).astype(float)
    bb = wm.paired_bootstrap_diff(wm.per_race_loss(m["race_id"].to_numpy(), m[f"p_{tgt}_b"].to_numpy(), y), wm.per_race_loss(m["race_id"].to_numpy(), m[f"p_{tgt}_v"].to_numpy(), y), n_boot=1000)
    out[f"{tgt}_base"], out[f"{tgt}_new"] = wm.binary_logloss(m[f"p_{tgt}_b"].to_numpy(), y), wm.binary_logloss(m[f"p_{tgt}_v"].to_numpy(), y)
    out[f"Δ{tgt}"], out[f"{tgt}_hi"] = bb["diff"], bb["hi"]
out["採用"] = bool(out["win_hi"] < 0 and out["Δtop2"] <= 0.0002 and out["Δtop3"] <= 0.0002)
cmp_ = pd.DataFrame([out]).set_index("variant")
display(cmp_.round(5)); cmp_.to_csv(OUT / "vs_nn_extras_ped.csv")
(OUT / "report.json").write_text(json.dumps({"git_commit": GIT_COMMIT, "best_params": BEST, "compare": out}, ensure_ascii=False, indent=1, default=str))
print("採用:", out["採用"])